# 10 — Multi-Agent Graph

**Learning objective:** model specialization and handoffs as one computation architecture inside an explicit graph—not as the definition of Graph Engineering.

The fake specialists here require no model or API key. The lesson is delegation, ownership, contracts, allowed routes, review, fallback, and termination.

## Mental model and topology

```mermaid
flowchart LR
    accTitle: Bounded multi-agent handoff graph
    accDescr: A manager selects one allowed specialist, an explicit handoff sends output to a reviewer, and deterministic quality policy completes or falls back.

    manager[Manager and allow-list] --> research_agent[Research agent]
    manager --> analysis_agent[Analysis agent]
    manager --> domain_agent[Domain agent]
    research_agent --> reviewer[Reviewer]
    analysis_agent --> reviewer
    domain_agent --> reviewer
    reviewer --> quality_gate{Quality gate}
    quality_gate --> complete([Complete])
    quality_gate --> fallback([Fallback])
```

## Choose the simplest justified architecture

| Option | Prefer it when | Avoid upgrading when |
| --- | --- | --- |
| Single function | Logic is deterministic and local | A prompt label adds no capability |
| Single agent | One context and tool set can complete the task | Roles use the same model, tools, and context |
| Subgraph | One responsibility has reusable internal flow | The workflow is tiny |
| Multi-agent graph | Roles need distinct tools, context boundaries, expertise, or valuable parallel specialization | Handoffs add latency but no capability |

Agent = computational actor. Graph = control structure. Multi-agent = specialization strategy.

## State, ownership, and policy

The manager owns `selected_agent`; a specialist owns `specialist_output`; the reviewer owns its assessment. `handoffs` records each transfer. A semantic proposal is accepted only if it maps to `ALLOWED_AGENTS`; deterministic policy substitutes a known fallback otherwise. Review has a finite complete/fallback outcome.

In [1]:
from graph_engineering.multi_agent import ALLOWED_AGENTS, build_multi_agent_graph

graph = build_multi_agent_graph()
result = graph.invoke({"task": "compare forecasts", "requested_role": "analysis"})
print("Allowed agents:", sorted(ALLOWED_AGENTS))
print("Selected:", result["selected_agent"])
print("Handoffs:", result["handoffs"])
print("Termination:", result["termination_reason"])

Allowed agents: ['analysis_agent', 'domain_agent', 'research_agent']
Selected: analysis_agent
Handoffs: [('manager', 'analysis_agent'), ('analysis_agent', 'reviewer'), ('reviewer', 'quality_gate')]
Termination: quality_reached


## Failure case: Supervisor Everywhere

A supervisor must not become one unconstrained LLM deciding every next node. It may suggest “this looks like research”; graph policy must still decide whether `research_agent` is allowed, whether budget remains, and which fallback is safe. Unknown routes, failed review, deadlines, and handoff loops all need deterministic bounds.

In [2]:
fallback_result = graph.invoke(
    {"task": "publish an answer", "requested_role": "unbounded_supervisor"}
)
print("Fallback used:", fallback_result["fallback_used"])
print("Selected safe agent:", fallback_result["selected_agent"])
print("First handoff:", fallback_result["handoffs"][0])

Fallback used: True
Selected safe agent: research_agent
First handoff: ('manager', 'research_agent')


## What to modify

Give one specialist a genuinely different contract or tool, then test its handoff. If the only difference is a role prompt, replace the agents with a simpler node or subgraph.

**Next:** [11 — Dynamic Routing](11_dynamic_routing.ipynb) varies work at runtime while preserving hard bounds.